# 01 - Read and join the Olist tables

**Job:** inspect every Task 1 table, aggregate one-to-many relations
before joining, and create exactly one row per order.

**Input:** PostgreSQL tables (preferred) or the equivalent Task 1 CSVs.

**Output:** `artifacts/01_ml_table.csv.gz` plus join/audit metadata.

In [1]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
print(f"Project root: {ROOT}")

Project root: C:\Users\Admin\Desktop\task_one\Brazilian-E-Commerce-Public-Dataset-by-Olist


## Load all nine tables

PostgreSQL is tried first because Task 1 placed the source tables in
the local database. A CSV fallback makes the notebook reproducible on
a machine where Docker is temporarily stopped; it is announced and
recorded in the artifacts.

In [2]:
TABLE_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}
configured_data_dir = os.getenv("OLIST_DATA_DIR")
raw_dir_candidates = (
    ([Path(configured_data_dir).expanduser()] if configured_data_dir else [])
    + [ROOT / "data" / "raw", ROOT / "data"]
)
RAW_DIR = next((
    candidate for candidate in raw_dir_candidates
    if all((candidate / filename).exists() for filename in TABLE_FILES.values())
), raw_dir_candidates[0])

tables = {}
data_source = "postgresql"
source_detail = ""
connection = None
try:
    import psycopg2

    connection = psycopg2.connect(
        host=os.getenv("PGHOST", os.getenv("POSTGRES_HOST", "localhost")),
        port=int(os.getenv("PGPORT", os.getenv("POSTGRES_PORT", "5432"))),
        dbname=os.getenv("PGDATABASE", os.getenv("POSTGRES_DB", "olist_db")),
        user=os.getenv("PGUSER", os.getenv("POSTGRES_USER", "olist_user")),
        password=os.getenv("PGPASSWORD", os.getenv("POSTGRES_PASSWORD", "olist_pass")),
        connect_timeout=3,
    )
    for table_name in TABLE_FILES:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            tables[table_name] = pd.read_sql_query(
                f'SELECT * FROM "{table_name}"', connection
            )
    source_detail = "All tables read from local PostgreSQL."
except Exception as exc:
    data_source = "csv_fallback"
    source_detail = (
        f"PostgreSQL unavailable ({type(exc).__name__}: {exc}); "
        "loaded the Task 1 CSV fallback."
    )
    print(source_detail)
    missing_files = [
        filename for filename in TABLE_FILES.values()
        if not (RAW_DIR / filename).exists()
    ]
    if missing_files:
        raise FileNotFoundError(f"Missing raw CSVs: {missing_files}") from exc
    tables = {
        name: pd.read_csv(RAW_DIR / filename, low_memory=False)
        for name, filename in TABLE_FILES.items()
    }
finally:
    if connection is not None:
        connection.close()

print(f"Data source used: {data_source}")
print(source_detail)

Data source used: postgresql
All tables read from local PostgreSQL.


## Light table audit

The audit checks row meaning, row counts, natural/declared keys, and
duplicate keys. Geolocation intentionally has repeated ZIP prefixes;
those rows are coordinate observations, not unique ZIP records.

In [3]:
TABLE_MEANING = {
    "customers": "one customer record used by one order",
    "geolocation": "one coordinate observation for a ZIP prefix",
    "order_items": "one item position inside an order",
    "order_payments": "one payment transaction/sequence for an order",
    "order_reviews": "one review row associated with an order",
    "orders": "one customer order",
    "products": "one product",
    "sellers": "one seller",
    "product_category_translation": "one Portuguese category translation",
}
KEY_COLUMNS = {
    "customers": ["customer_id"],
    "geolocation": None,
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_translation": ["product_category_name"],
}

audit_rows = []
for name, frame in tables.items():
    keys = KEY_COLUMNS[name]
    duplicate_keys = None if keys is None else int(frame.duplicated(keys).sum())
    audit_rows.append({
        "table": name,
        "row_meaning": TABLE_MEANING[name],
        "rows": len(frame),
        "columns": frame.shape[1],
        "key": "(observations; ZIP repeats)" if keys is None else ", ".join(keys),
        "duplicate_key_rows": duplicate_keys,
    })
table_audit = pd.DataFrame(audit_rows).sort_values("table").reset_index(drop=True)
display(table_audit)

# Inspect every table on its own, as required by Task 2.
for name in TABLE_FILES:
    frame = tables[name]
    print(f"\n{name}: shape={frame.shape}; one row = {TABLE_MEANING[name]}")
    display(frame.head(2))

key_failures = table_audit[table_audit["duplicate_key_rows"].fillna(0).gt(0)]
assert key_failures.empty, f"Unexpected duplicate table keys:\n{key_failures}"

,table,row_meaning,rows,columns,key,duplicate_key_rows
0,customers,one customer record used by one order,99441,5,customer_id,0.0
1,geolocation,one coordinate observation for a ZIP prefix,1000163,5,(observations; ZIP repeats),NaN
2,order_items,one item position inside an order,112650,7,"order_id, order_item_id",0.0
3,order_payments,one payment transaction/sequence for an order,103886,5,"order_id, payment_sequential",0.0
4,order_reviews,one review row associated with an order,99224,7,"review_id, order_id",0.0
5,orders,one customer order,99441,8,order_id,0.0
6,product_category_translation,one Portuguese category translation,71,2,product_category_name,0.0
7,products,one product,32951,9,product_id,0.0
8,sellers,one seller,3095,4,seller_id,0.0



customers: shape=(99441, 5); one row = one customer record used by one order


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP



geolocation: shape=(1000163, 5); one row = one coordinate observation for a ZIP prefix


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0



sellers: shape=(3095, 4); one row = one seller


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP



product_category_translation: shape=(71, 2); one row = one Portuguese category translation


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories


## Aggregate before joining

`order_items` and `order_payments` contain multiple rows per order.
They are enriched and collapsed first. Reviews are audited but are
deliberately excluded: review content and score exist after delivery
and would leak future information into a delivery-time prediction.

In [4]:
def mode_or_missing(series: pd.Series):
    values = series.dropna()
    if values.empty:
        return pd.NA
    modes = values.mode()
    return modes.iloc[0] if not modes.empty else values.iloc[0]


orders = tables["orders"].copy()
customers = tables["customers"].copy()
items = tables["order_items"].copy()
payments = tables["order_payments"].copy()
products = tables["products"].copy()
sellers = tables["sellers"].copy()
translations = tables["product_category_translation"].copy()
geolocation = tables["geolocation"].copy()

# The original Kaggle CSV spells these two fields as "lenght";
# Task 1's PostgreSQL schema corrected them to "length".
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

for column in [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

products = products.merge(
    translations, on="product_category_name", how="left", validate="many_to_one"
)
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

item_enriched = (
    items.merge(products, on="product_id", how="left", validate="many_to_one")
    .merge(sellers, on="seller_id", how="left", validate="many_to_one")
)
item_agg = item_enriched.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
    max_item_price=("price", "max"),
    product_category_count=("product_category_name", "nunique"),
    product_category_mode=("product_category_name_english", mode_or_missing),
    product_name_length_mean=("product_name_length", "mean"),
    product_description_length_mean=("product_description_length", "mean"),
    product_photos_qty_mean=("product_photos_qty", "mean"),
    product_weight_g_mean=("product_weight_g", "mean"),
    product_volume_cm3_mean=("product_volume_cm3", "mean"),
    seller_state_mode=("seller_state", mode_or_missing),
    seller_zip_code_prefix_mode=("seller_zip_code_prefix", mode_or_missing),
    shipping_limit_date_max=("shipping_limit_date", "max"),
)

payment_agg = payments.groupby("order_id", as_index=False).agg(
    payment_count=("payment_sequential", "count"),
    payment_value_total=("payment_value", "sum"),
    max_payment_installments=("payment_installments", "max"),
    payment_type_count=("payment_type", "nunique"),
    payment_type_mode=("payment_type", mode_or_missing),
)

assert item_agg["order_id"].is_unique
assert payment_agg["order_id"].is_unique
print(f"Items: {len(items):,} rows -> {len(item_agg):,} order aggregates")
print(f"Payments: {len(payments):,} rows -> {len(payment_agg):,} order aggregates")

Items: 112,650 rows -> 98,666 order aggregates
Payments: 103,886 rows -> 99,440 order aggregates


## Join one row per order and add geographic distance

In [5]:
geo_agg = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geo_lat=("geolocation_lat", "median"),
    geo_lng=("geolocation_lng", "median"),
    geo_observations=("geolocation_lat", "size"),
)
customer_geo = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "customer_zip_code_prefix",
    "geo_lat": "customer_lat",
    "geo_lng": "customer_lng",
    "geo_observations": "customer_geo_observations",
})
seller_geo = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix_mode",
    "geo_lat": "seller_lat",
    "geo_lng": "seller_lng",
    "geo_observations": "seller_geo_observations",
})

ml_table = (
    orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(item_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payment_agg, on="order_id", how="left", validate="one_to_one")
    .merge(customer_geo, on="customer_zip_code_prefix", how="left", validate="many_to_one")
    .merge(seller_geo, on="seller_zip_code_prefix_mode", how="left", validate="many_to_one")
)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

ml_table["customer_seller_distance_km"] = haversine_km(
    ml_table["customer_lat"], ml_table["customer_lng"],
    ml_table["seller_lat"], ml_table["seller_lng"],
)

assert len(ml_table) == len(orders)
assert ml_table["order_id"].is_unique
print(f"ML table shape: {ml_table.shape}; unique orders: {ml_table['order_id'].nunique():,}")
display(ml_table.head(3))

ML table shape: (99441, 41); unique orders: 99,441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,product_count,seller_count,total_price,total_freight,avg_item_price,max_item_price,product_category_count,product_category_mode,product_name_length_mean,product_description_length_mean,product_photos_qty_mean,product_weight_g_mean,product_volume_cm3_mean,seller_state_mode,seller_zip_code_prefix_mode,shipping_limit_date_max,payment_count,payment_value_total,max_payment_installments,payment_type_count,payment_type_mode,customer_lat,customer_lng,customer_geo_observations,seller_lat,seller_lng,seller_geo_observations,customer_seller_distance_km
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,29.99,29.99,1.0,housewares,40.0,268.0,4.0,500.0,1976.0,SP,9350,2017-10-06 11:07:15,3.0,38.71,1.0,2.0,voucher,-23.576170,-46.587276,24.0,-23.681180,-46.444127,207.0,18.681737
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,118.70,118.70,1.0,perfumery,29.0,178.0,1.0,400.0,4693.0,SP,31570,2018-07-30 03:24:27,1.0,141.46,1.0,1.0,boleto,-12.126651,-45.008162,19.0,-19.807013,-43.980966,71.0,861.036556
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,159.90,159.90,1.0,auto,46.0,232.0,1.0,420.0,9576.0,SP,14840,2018-08-13 08:55:23,1.0,179.12,3.0,1.0,credit_card,-16.744472,-48.514624,25.0,-21.364020,-48.228831,170.0,514.547851


## Save the step-1 artifact

In [6]:
output_path = ARTIFACT_DIR / "01_ml_table.csv.gz"
ml_table.to_csv(output_path, index=False, compression="gzip")
table_audit.to_csv(ARTIFACT_DIR / "01_table_audit.csv", index=False)
join_summary = {
    "data_source": data_source,
    "source_detail": source_detail,
    "source_order_rows": int(len(orders)),
    "output_rows": int(len(ml_table)),
    "output_columns": int(ml_table.shape[1]),
    "unique_orders": int(ml_table["order_id"].nunique()),
    "reviews_excluded_as_future_information": True,
    "items_aggregated_before_join": True,
    "payments_aggregated_before_join": True,
}
(ARTIFACT_DIR / "01_join_summary.json").write_text(
    json.dumps(join_summary, indent=2), encoding="utf-8"
)
assert output_path.exists() and output_path.stat().st_size > 0
print(f"Saved {output_path.relative_to(ROOT)} ({output_path.stat().st_size / 1e6:.1f} MB)")

Saved artifacts\01_ml_table.csv.gz (19.2 MB)
